# Reversible Product Inhibition Example

This notebook is a researcher-facing exploratory example for FungMod's generic reversible product-inhibition modifier. It is not an empirical validation, calibration, toxicity, uptake, secretion, biomass, whole-fungus physiology, or multi-product inhibition example. The only inhibition law demonstrated here is the implemented configured multiplier `1 / (1 + P / K_i)` with an explicit product state and positive `K_i` fixture.

In [ ]:
import json
import os
from pathlib import Path

from fungal_model import virtual_experiment
from fungal_model.examples import prepare_reversible_product_inhibition_example_registry

OUTPUT_ROOT = Path(os.environ.get("FUNGMOD_NOTEBOOK_OUTPUT_ROOT", "outputs/notebooks"))
OUTPUT_DIR = OUTPUT_ROOT / "12_reversible_product_inhibition_example"
BASE_REGISTRY = Path("data_registry/registry_index.yml")
EXAMPLE_REGISTRY = prepare_reversible_product_inhibition_example_registry(
    OUTPUT_ROOT / "12_reversible_product_inhibition_registry",
    source_registry=BASE_REGISTRY,
)


Build two virtual experiments through researcher-facing names: the baseline registry has no product-inhibition modifier, while the copied example registry adds one explicit exploratory `K_i` fixture to the existing BIO-002 enzyme-chain template. The fixture is provenance-labelled and not validation data.

In [ ]:
uninhibited_study = virtual_experiment(
    fungi="generic cellulase source",
    substrates="cellulose film",
    environments="30 C pH 5 assay",
    registry=BASE_REGISTRY,
)
inhibited_study = virtual_experiment(
    fungi="generic cellulase source",
    substrates="cellulose film",
    environments="30 C pH 5 assay",
    registry=EXAMPLE_REGISTRY,
)

[(report.status, report.required_processes) for report in inhibited_study.preflight(mode="exploratory")]


Run one deterministic exploratory sample for each registry. These outputs demonstrate configured mechanics only; they are not evidence that the chosen `K_i` applies to a biological system.

In [ ]:
uninhibited_result = uninhibited_study.simulate(
    mode="exploratory",
    n_samples=1,
    seed=31,
    output_dir=OUTPUT_DIR / "uninhibited",
    quicklook=False,
)
inhibited_result = inhibited_study.simulate(
    mode="exploratory",
    n_samples=1,
    seed=31,
    output_dir=OUTPUT_DIR / "inhibited",
    quicklook=False,
)

sorted(Path(inhibited_result.output_directory).glob("*.csv"))[:5]


Inspect `mechanism_summary.csv` and configured metadata first. The active modifier row is where the implemented law, product state, `K_i` symbol, maturity, assumptions, limitations, and provenance are visible.

In [ ]:
mechanism_rows = inhibited_result.mechanism_summary()
modifier_rows = [row for row in mechanism_rows if row["mechanism_id"] == "product_inhibition"]
metadata_path = next(Path(inhibited_result.output_directory).glob("*/sample_0000/bundle/configured_metadata.json"))
configured_metadata = json.loads(metadata_path.read_text(encoding="utf-8"))

modifier_rows, configured_metadata["configured_process_modifiers"]


The standard result tables can compare outputs without adding notebook-only scientific logic. Here the inhibited run has lower final product concentration for the same explicit example fixture, which is a deterministic software demonstration of the configured multiplier.

In [ ]:
uninhibited_metrics = {row["metric"]: row for row in uninhibited_result.final_metrics()}
inhibited_metrics = {row["metric"]: row for row in inhibited_result.final_metrics()}

comparison = {
    "uninhibited_final_product_concentration": float(uninhibited_metrics["final_product_concentration"]["value"]),
    "inhibited_final_product_concentration": float(inhibited_metrics["final_product_concentration"]["value"]),
    "inhibition_law": modifier_rows[0]["equation_or_law"],
    "limitations": modifier_rows[0]["limitations"],
}
comparison
